# NB07 · Commercial mortgage loans

**Project:** multiyear-credit-engine
**Job requirement covered:** Commercial mortgage loans; asset and risk characteristics.
**Engine module:** `creditengine.assets.cre`
**Assumptions:** A-07, A-09, A-14
**Theory:** Merton applied to property value; NAIC Life RBC instructions (Figures 2, 4, 5)
**Status:** ✅ Developed — every function is built from scratch in this notebook

## Objective

How does a commercial mortgage book lose money over five years — when does a loan default (can't pay from income, or can't refinance at maturity), how much is lost when it does, and how does the book's NAIC capital category move as property values and income change with the cycle?

## Results at a glance

| Question | Answer |
|---|---|
| What does the book look like? | 60 loans, USD 170m, 42% apartments; average DSCR about 2.0 and LTV about 62%; office weakest (DSCR 1.67, LTV 78%) |
| How often do loans default in five years? | **3.1%** of loans on average; office 5.2%, retail 0.8% |
| How do they default? | **95% at the balloon** — refinancing fails at maturity. Term default is rare because it needs *both* DSCR < 1 and LTV > 1 |
| How much is lost? | Mean LGD **29%**; 12% of defaults lose nothing. Five-year expected loss **0.88%** of the book, ES 99.5% **9.7%** |
| How does capital move? | Year-0 RBC **USD 1.9m** (1.1% of balance), 81% of the book in CM1. In the worst 0.5% of paths the share in CM3 or worse rises from 5% to **56%** by year 5 |
| Do the settings fit public data? | Simulated property-value swings match the BIS CRE price index (std 9.2% vs 9.6%); the 1-in-36-year loss rate (2.1%) is close to the worst bank charge-off year since 1991 (2.35%) |
| Is the NAIC grid coded right? | Every (DSCR, LTV) point falls in exactly one category, for both Figures 4 and 5 |

## Key terms and symbols

| Term | Meaning |
|---|---|
| $V_t$ | Property value in year $t$ |
| $NOI_t$ | Net operating income: rent minus operating costs, per year |
| $B_t$ | Loan balance still owed in year $t$ |
| Debt service | Annual loan payment (interest + principal), fixed at origination |
| $DSCR_t$ | Debt service coverage ratio $= NOI_t / \text{debt service}$: can the building pay the loan? |
| $LTV_t$ | Loan to value $= B_t / V_t$: is there equity left above the loan? |
| Balloon | The balance still owed at maturity, which must be refinanced or repaid |
| $\kappa$ | Selling and workout costs as a share of value (legal costs, time, fire-sale discount) |
| $Z_t$ | Commercial real estate cycle factor, shared by all properties (low = bad year) |
| CM1–CM7 | NAIC commercial mortgage categories; CM1–CM5 for loans in good standing, set by DSCR and LTV |

## Inputs

| Data | File | Used for |
|---|---|---|
| Stylised loan book: 60 loans, USD 170m (A-09) | built in section 1, settings from `configs/portfolios/commercial_mortgages.yaml` | the portfolio |
| FRED CRE series: delinquency, charge-offs (banks), CRE prices (BIS) | `data/raw/fred/commercial_real_estate/` | calibration check (section 6) |
| NAIC CM grid and factors | NAIC Life RBC instructions, Figures 2, 4 and 5 — typed in section 5 | CM category and capital |

Loan-level public data (CMBS Form ABS-EE, Fannie Mae multifamily MFLPD) is planned for v0.3; this notebook uses a stylised book with averages close to a large life insurer's public disclosure (apartment-heavy, average DSCR ≈ 2.0, LTV ≈ 60%, a weaker office pocket).

## Outputs

- `data/interim/cre_loans_stylised.parquet` — the loan book
- `data/processed/cre_parameters.yaml` — property, loan and cycle settings used
- `reports/figures/cre_dscr_ltv.png` — DSCR and LTV of every loan against the default triggers
- `reports/figures/cre_cm_migration.png` — CM category mix of the book, year by year

## 0. Setup

Settings for property behaviour (A-07) and the cycle, all in one place. The seed is fixed for repeatable results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import norm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
rng = np.random.default_rng(11)
pd.set_option("display.float_format", "{:,.3f}".format)

SETTINGS = {                              # A-07: illustrative, declared, not fitted
    "value_drift": 0.02, "value_vol": 0.15, "value_rho": 0.35,     # property value
    "noi_growth": 0.02, "noi_vol": 0.08, "noi_rho": 0.30,          # net operating income
    "selling_cost": 0.25,                                          # kappa
    "prepay_rate": 0.03,                                           # per year
    "refi_rate": 0.06, "refi_ltv_max": 1.00, "refi_dscr_min": 1.00,  # balloon refinancing test
    "phi": 0.50,                                                   # persistence of the CRE cycle
}
HORIZON, N_PATHS = 5, 20_000

## 1. Loan-level data: DSC, LTV, property type, maturity

$$\text{debt service} = B_0 \frac{r}{1 - (1 + r)^{-n}}$$

Level annual payment that repays the loan over $n$ years of amortisation at rate $r$.

$$B_t = B_0 (1 + r)^t - \text{debt service} \times \frac{(1 + r)^t - 1}{r}$$

Balance still owed after $t$ payments. With 25-year amortisation and 1–10-year maturities, most of the loan is still owed at the balloon.

### 1.1 Loan arithmetic

In [ ]:
def annual_payment(balance, rate, amort_years):
    return balance * rate / (1 - (1 + rate) ** (-amort_years))

def balance_after(balance, rate, amort_years, years):
    growth = (1 + rate) ** years
    return balance * growth - annual_payment(balance, rate, amort_years) * (growth - 1) / rate

pd.Series({t: balance_after(100.0, 0.05, 25, t) for t in [0, 1, 5, 10]}, name="balance left of 100").to_frame().T

### 1.2 The stylised book (A-09)

60 loans, USD 170m. Loan sizes are lognormal; LTV and DSCR are drawn around the book averages, with office loans drawn weaker (+12 points LTV, −0.45 DSCR) to mirror the post-2020 office pocket.

In [ ]:
BOOK = {"total": 170e6, "n_loans": 60, "rate": 0.05, "amort": 25, "max_maturity": 10,
        "mix": {"apartment": 0.42, "industrial": 0.22, "retail": 0.22, "office": 0.12, "hotel": 0.02},
        "ltv_mean": 0.60, "ltv_sd": 0.12, "dscr_mean": 2.0, "dscr_sd": 0.45,
        "office_ltv_add": 0.12, "office_dscr_cut": 0.45}

def make_book(p, rng):
    types = np.repeat(list(p["mix"]), np.round(np.array(list(p["mix"].values())) * p["n_loans"]).astype(int))
    size = rng.lognormal(0.0, 0.6, types.size)
    balance = np.zeros(types.size)
    for name, share in p["mix"].items():                      # each type gets its share of the total
        m = types == name
        balance[m] = size[m] / size[m].sum() * share * p["total"]
    office = types == "office"
    ltv = np.clip(rng.normal(p["ltv_mean"], p["ltv_sd"], types.size) + office * p["office_ltv_add"], 0.30, 0.95)
    dscr = np.clip(rng.normal(p["dscr_mean"], p["dscr_sd"], types.size) - office * p["office_dscr_cut"], 1.10, 3.50)
    debt_service = annual_payment(balance, p["rate"], p["amort"])
    return pd.DataFrame({"type": types, "balance": balance, "value": balance / ltv, "noi": dscr * debt_service,
                         "debt_service": debt_service, "dscr": dscr, "ltv": ltv,
                         "maturity": rng.integers(1, p["max_maturity"], endpoint=True, size=types.size)})

book = make_book(BOOK, rng)
book.groupby("type").agg(loans=("balance", "size"), balance_m=("balance", lambda b: b.sum() / 1e6),
                         avg_dscr=("dscr", "mean"), avg_ltv=("ltv", "mean"), avg_maturity=("maturity", "mean"))

**Reading the result:** an apartment-heavy, low-leverage book — average DSCR around 2 means income covers the loan payment twice. Office is the weak pocket (DSCR 1.67, LTV 78%), and the single hotel loan is the most leveraged (LTV 91%) with a balloon in two years.

### 1.3 Figure: every loan against the default triggers

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, g in book.groupby("type"):
    ax.scatter(g["ltv"], g["dscr"], s=g["balance"] / 1e5, alpha=0.7, label=name)
ax.axhline(1.0, color="red", lw=1, ls="--"); ax.axvline(1.0, color="red", lw=1, ls="--")
ax.text(1.01, 0.6, "term default zone:\nDSCR < 1 and LTV > 1", color="red", fontsize=8)
ax.set(xlim=(0.2, 1.3), ylim=(0.4, 3.7), xlabel="LTV (loan / value)", ylabel="DSCR (income / payment)",
       title="Stylised commercial mortgage book today (bubble size = balance)")
ax.legend(fontsize=8)
fig.savefig(ROOT / "reports" / "figures" / "cre_dscr_ltv.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Property value and NOI paths driven by the cycle factor

**Merton on the building:** the property plays the role of the firm's assets, and the loan is the debt.

$$\ln V_t = \ln V_{t-1} + \mu_V - \tfrac12\sigma_V^2 + \sigma_V\left(\sqrt{\rho_V} Z_t + \sqrt{1-\rho_V} e_t\right)$$

Each year the value grows by its drift plus a shock: part shared by all properties ($Z_t$), part specific to the building ($e_t$).

$$\ln NOI_t = \ln NOI_{t-1} + g - \tfrac12\sigma_N^2 + \sigma_N\left(\sqrt{\rho_N} Z_t + \sqrt{1-\rho_N} u_t\right)$$

Income follows the same pattern, with smaller swings.

$$Z_t = \phi Z_{t-1} + \sqrt{1-\phi^2} \eta_t$$

The CRE cycle is persistent: a bad year tends to follow a bad year. In the whole-book model $Z_t = \sqrt{\beta} G_t + \sqrt{1-\beta} H_t$ ties it half to the global credit cycle ($\beta = 0.5$, A-14).

| Symbol | Meaning | Value (A-07) |
|---|---|---|
| $\mu_V, \sigma_V, \rho_V$ | Value drift, volatility, share of variance from the cycle | 2%, 15%, 0.35 |
| $g, \sigma_N, \rho_N$ | NOI growth, volatility, share from the cycle | 2%, 8%, 0.30 |
| $e_t, u_t, \eta_t$ | Independent standard normal shocks | — |
| $\phi$ | Cycle persistence | 0.5 |

### 2.1 Cycle paths

In [ ]:
def cycle_paths(n_paths, years, phi, rng):
    # AR(1) factor paths, stationary standard normal, shape (paths, years)
    z = np.empty((n_paths, years))
    z[:, 0] = rng.standard_normal(n_paths)
    for t in range(1, years):
        z[:, t] = phi * z[:, t - 1] + np.sqrt(1 - phi**2) * rng.standard_normal(n_paths)
    return z

Z = cycle_paths(N_PATHS, HORIZON, SETTINGS["phi"], rng)
pd.DataFrame({"mean": Z.mean(axis=0), "std": Z.std(axis=0),
              "corr with previous year": [np.nan] + [np.corrcoef(Z[:, t], Z[:, t - 1])[0, 1] for t in range(1, HORIZON)]},
             index=[f"year {t}" for t in range(1, HORIZON + 1)])

### 2.2 One step of value and NOI

In [ ]:
def step(log_x, drift, vol, rho, z_t, shock):
    # One year of a lognormal path: drift, shared cycle shock, own shock
    return log_x + drift - 0.5 * vol**2 + vol * (np.sqrt(rho) * z_t + np.sqrt(1 - rho) * shock)

s = SETTINGS
for name, z in {"good year (Z = +2)": 2.0, "typical (Z = 0)": 0.0, "bad year (Z = -2)": -2.0}.items():
    v = np.exp(step(0.0, s["value_drift"], s["value_vol"], s["value_rho"], z, 0.0)) - 1
    n = np.exp(step(0.0, s["noi_growth"], s["noi_vol"], s["noi_rho"], z, 0.0)) - 1
    print(f"{name:20s} value {v:+.1%}   NOI {n:+.1%}   (building with no own shock)")

## 3. Term default (DSC < 1) and balloon default (value < balance)

$$\text{term default in year } t: \quad DSCR_t < 1 \quad \text{and} \quad LTV_t > 1$$

**Double trigger.** The building cannot pay the loan from its income, **and** the owner has no equity left to protect — so walking away is rational. With equity left, the owner keeps paying or sells.

$$\text{balloon default at maturity: } \quad LTV_T > 1 \quad \text{or} \quad \frac{NOI_T}{\text{new debt service}} < 1$$

**Refinancing fails.** At maturity the remaining balance must be refinanced. A lender needs the loan to fit under the value and the income to cover the new payment at today's rate (6%, A-07).

A loan leaves the book when it defaults, matures, or prepays (3% a year).

### 3.1 Simulate the book

In [ ]:
def simulate(book, Z, s, rng, p=BOOK):
    n_paths, years = Z.shape
    n = len(book)
    log_v = np.tile(np.log(book["value"].to_numpy()), (n_paths, 1))
    log_n = np.tile(np.log(book["noi"].to_numpy()), (n_paths, 1))
    ds, mat = book["debt_service"].to_numpy(), book["maturity"].to_numpy()
    alive = np.ones((n_paths, n), bool)
    out = {"loss": np.zeros((n_paths, years, n)), "default_year": np.zeros((n_paths, n), int),
           "balloon": np.zeros((n_paths, n), bool), "dscr": np.zeros((n_paths, years, n)),
           "ltv": np.zeros((n_paths, years, n)), "balance": np.zeros((n_paths, years + 1, n)),
           "alive": np.zeros((n_paths, years + 1, n), bool), "value_change": np.zeros((n_paths, years))}
    out["balance"][:, 0], out["alive"][:, 0] = book["balance"].to_numpy(), True
    for t in range(1, years + 1):
        z = Z[:, t - 1:t]
        new_v = step(log_v, s["value_drift"], s["value_vol"], s["value_rho"], z, rng.standard_normal((n_paths, n)))
        out["value_change"][:, t - 1] = np.exp(new_v - log_v).mean(axis=1) - 1     # book-wide value index
        log_v = new_v
        log_n = step(log_n, s["noi_growth"], s["noi_vol"], s["noi_rho"], z, rng.standard_normal((n_paths, n)))
        value, noi = np.exp(log_v), np.exp(log_n)
        bal = balance_after(book["balance"].to_numpy(), p["rate"], p["amort"], t)
        dscr, ltv = noi / ds, bal / value
        term = (dscr < 1) & (ltv > 1)
        balloon = (mat == t) & ((ltv > s["refi_ltv_max"]) | (noi / annual_payment(bal, s["refi_rate"], p["amort"]) < s["refi_dscr_min"]))
        default = alive & (term | balloon)
        out["loss"][:, t - 1] = np.where(default, np.maximum(bal - (1 - s["selling_cost"]) * value, 0), 0)
        out["default_year"][default] = t
        out["balloon"] |= default & balloon & ~term
        prepaid = alive & ~default & (rng.random((n_paths, n)) < s["prepay_rate"])
        alive &= ~(default | prepaid | (mat == t))
        out["dscr"][:, t - 1], out["ltv"][:, t - 1] = dscr, ltv
        out["balance"][:, t], out["alive"][:, t] = np.where(alive, bal, 0), alive
    return out

sim = simulate(book, Z, SETTINGS, rng)
defaulted = sim["default_year"] > 0
print(f"5-year default rate: {defaulted.mean():.2%} of loans  |  "
      f"balloon share of defaults: {sim['balloon'][defaulted].mean():.0%}")

### 3.2 Who defaults, and how

In [ ]:
by_type = pd.DataFrame({"5-yr default %": 100 * pd.DataFrame(defaulted, columns=book["type"]).T.groupby(level=0).mean().mean(axis=1),
                        "balloon share of defaults %": 100 * pd.Series(
                            {k: sim["balloon"][:, (book["type"] == k).to_numpy()][defaulted[:, (book["type"] == k).to_numpy()]].mean()
                             for k in BOOK["mix"]})})
by_type.loc["whole book"] = [100 * defaulted.mean(), 100 * sim["balloon"][defaulted].mean()]
by_type

**Reading the result:** about 3% of loans default within five years, and almost all of them **at the balloon**. With income at twice the payment, a loan rarely fails the double trigger while it is running; the danger is the maturity date, when the full balance must be refinanced at a higher rate (6% vs 5%) against a value that may have fallen. That is the "maturity wall" that office loans faced after 2022. The hotel loan (LTV 91%, two years to maturity) defaults in about a quarter of paths; retail, with the strongest coverage, in under 1%.

### 3.3 Check against the Merton closed form

For an interest-only loan with no income test and no cycle persistence, balloon default is exactly a Merton default: the value at maturity falls below the balance.

$$PD = N(-DD), \qquad DD = \frac{\ln(V_0/B) + (\mu_V - \tfrac12\sigma_V^2) T}{\sigma_V\sqrt{T}}$$

Test case: LTV 70%, $T = 5$ years, 1,000,000 simulated value paths.

In [ ]:
V0, B, T5, n_check = 100.0, 70.0, 5, 1_000_000
dd = (np.log(V0 / B) + (s["value_drift"] - 0.5 * s["value_vol"]**2) * T5) / (s["value_vol"] * np.sqrt(T5))
z_iid = cycle_paths(n_check, T5, 0.0, rng)                    # phi = 0: independent years
log_v = np.full(n_check, np.log(V0))
for t in range(T5):
    log_v = step(log_v, s["value_drift"], s["value_vol"], s["value_rho"], z_iid[:, t], rng.standard_normal(n_check))
pd_closed, pd_sim = norm.cdf(-dd), (np.exp(log_v) < B).mean()
print(f"closed form {pd_closed:.2%}   simulated {pd_sim:.2%}   (standard error {np.sqrt(pd_sim * (1 - pd_sim) / n_check):.2%})")

## 4. Loss severity from property value

$$\text{loss} = \max\big(B - (1-\kappa) V, 0\big), \qquad LGD = \frac{\text{loss}}{B}$$

The lender takes the building, sells it, and loses whatever the net sale proceeds don't cover. $\kappa$ = 25% covers legal costs, time and the fire-sale discount.

*Example:* balance 70, value 60 → proceeds $0.75 \times 60 = 45$ → loss 25 → LGD 36%.

### 4.1 Severity of the defaults

In [ ]:
loan_loss = sim["loss"].sum(axis=1)                                   # (paths, loans)
bal0 = book["balance"].to_numpy()
default_balance = np.where(defaulted, balance_after(bal0, BOOK["rate"], BOOK["amort"], sim["default_year"]), 0)
lgd = loan_loss[defaulted] / default_balance[defaulted]
pd.Series({"mean LGD": lgd.mean(), "median LGD": np.median(lgd), "share of defaults with zero loss": (lgd == 0).mean(),
           "90th percentile LGD": np.quantile(lgd, 0.9)}).to_frame("value").T

### 4.2 Five-year loss distribution of the book

Expected loss (average path), VaR (loss exceeded in 0.5% of paths) and expected shortfall (average of that worst 0.5%).

In [ ]:
total = sim["loss"].sum(axis=(1, 2))
q = 0.995
var = np.quantile(total, q)
summary = pd.Series({"expected loss": total.mean(), "VaR 99.5%": var, "ES 99.5%": total[total >= var].mean()})
pd.DataFrame({"USD m": summary / 1e6, "% of book": 100 * summary / bal0.sum()})

**Reading the result:** when a loan defaults the lender typically loses about 30% of the balance; about one default in eight loses nothing because the building still sells for more than the loan after costs. Over five years the book expects to lose 0.9% of its balance, but in the worst 0.5% of paths it loses close to 10% — the tail is about eleven times the average, because property values and refinancing fail together in a bad cycle.

## 5. DSC/LTV → CM1–CM7 category and its migration

The NAIC capital charge for a commercial mortgage depends on its category, set each year from the loan's DSCR and LTV. CM6 (90 days overdue) and CM7 (in foreclosure) apply to loans not in good standing; in this model a defaulted loan leaves the book, so the live book uses CM1–CM5.

$$\text{RBC}_t = \sum_{\text{live loans}} B_t \times \text{factor}(CM_t)$$

### 5.1 The NAIC grid, typed from the instructions

Each row is one line of NAIC Life RBC Figure 4 (office, industrial, retail, multifamily) or Figure 5 (hotel and specialty): a DSCR band and an LTV band. Factors from Figure 2.

*Note on Figure 5:* the last CM5 line prints as "1.10 ≤ DSC and 90% ≤ LTV". Read literally it overlaps CM3 and leaves loans with DSC < 0.90 and LTV ≥ 90% without a category. Read as DSC < 1.10, the grid covers every loan exactly once — the check in 5.2 confirms it.

In [ ]:
INF = np.inf
GRID = pd.DataFrame([  # figure, category, DSC from, DSC to, LTV from, LTV to   (lower bound included, upper excluded)
    (4, 1, 1.50, INF, 0.00, 0.85),
    (4, 2, 0.00, 1.50, 0.00, 0.55), (4, 2, 0.95, 1.50, 0.55, 0.75), (4, 2, 1.15, 1.50, 0.75, 1.00),
    (4, 2, 1.50, INF, 0.85, 1.00), (4, 2, 1.75, INF, 1.00, INF),
    (4, 3, 0.00, 0.95, 0.55, 0.85), (4, 3, 0.95, 1.15, 0.75, 1.00), (4, 3, 1.15, 1.75, 1.00, INF),
    (4, 4, 0.00, 0.95, 0.85, 1.05), (4, 4, 0.95, 1.15, 1.00, INF),
    (4, 5, 0.00, 0.95, 1.05, INF),
    (5, 1, 1.85, INF, 0.00, 0.60),
    (5, 2, 1.45, 1.85, 0.00, 0.70), (5, 2, 1.85, INF, 0.60, 1.15),
    (5, 3, 0.90, 1.45, 0.00, 0.80), (5, 3, 1.45, 1.85, 0.70, INF), (5, 3, 1.85, INF, 1.15, INF),
    (5, 4, 0.00, 0.90, 0.00, 0.90), (5, 4, 0.90, 1.10, 0.80, 0.90), (5, 4, 1.10, 1.45, 0.80, INF),
    (5, 5, 0.00, 1.10, 0.90, INF),
], columns=["figure", "cm", "dsc_from", "dsc_to", "ltv_from", "ltv_to"])
CM_FACTOR = {1: 0.0090, 2: 0.0175, 3: 0.0300, 4: 0.0500, 5: 0.0750}   # Figure 2, pre-tax, in good standing

def cm_category(dscr, ltv, hotel):
    # CM1..CM5 for each loan: the grid row whose DSC and LTV bands contain it
    dscr, ltv, hotel = np.broadcast_arrays(np.asarray(dscr, float), np.asarray(ltv, float), np.asarray(hotel, bool))
    cat = np.zeros(dscr.shape, np.int8)
    for row in GRID.itertuples():
        inside = (dscr >= row.dsc_from) & (dscr < row.dsc_to) & (ltv >= row.ltv_from) & (ltv < row.ltv_to)
        cat[inside & (hotel == (row.figure == 5))] = row.cm
    return cat

book["cm"] = cm_category(book["dscr"], book["ltv"], book["type"] == "hotel")
book.groupby("cm")["balance"].sum().div(book["balance"].sum()).rename("share of balance").to_frame().T

### 5.2 Check: the grid covers every loan exactly once (V-12)

Lay a fine grid of 600 × 600 (DSCR, LTV) points over DSCR 0–3 and LTV 0–1.5, and count how many rows of each figure claim each point. Every point must be claimed exactly once.

In [ ]:
d, v = np.meshgrid(np.linspace(0, 3, 600), np.linspace(0, 1.5, 600))
coverage = {}
for fig_no in (4, 5):
    hits = sum(((d >= r.dsc_from) & (d < r.dsc_to) & (v >= r.ltv_from) & (v < r.ltv_to)).astype(int)
               for r in GRID[GRID["figure"] == fig_no].itertuples())
    coverage[f"Figure {fig_no}"] = {"points": hits.size, "claimed once": int((hits == 1).sum()),
                                    "gaps": int((hits == 0).sum()), "overlaps": int((hits > 1).sum())}
spot = {(2.0, 0.60): 1, (1.30, 0.50): 2, (1.30, 0.80): 2, (1.00, 0.90): 3, (0.90, 0.95): 4, (0.90, 1.10): 5}
spot_ok = all(cm_category(a, b, False) == c for (a, b), c in spot.items())
print("spot checks against Figure 4 lines:", "all correct" if spot_ok else "MISMATCH")
pd.DataFrame(coverage).T

### 5.3 CM migration and the capital path

In [ ]:
hotel = (book["type"] == "hotel").to_numpy()
cm_path = np.zeros((N_PATHS, HORIZON + 1, len(book)), np.int8)
cm_path[:, 0] = book["cm"].to_numpy()
for t in range(1, HORIZON + 1):
    cm_path[:, t] = np.where(sim["alive"][:, t], cm_category(sim["dscr"][:, t - 1], sim["ltv"][:, t - 1], hotel), 0)

factor = np.vectorize(lambda c: CM_FACTOR.get(int(c), 0.0))(cm_path)
rbc = (factor * sim["balance"]).sum(axis=2)                                    # (paths, years + 1)
mix = pd.DataFrame({t: [(sim["balance"][:, t] * (cm_path[:, t] == c)).sum() / sim["balance"][:, t].sum() for c in range(1, 6)]
                    for t in range(HORIZON + 1)}, index=[f"CM{c}" for c in range(1, 6)])
rbc_table = pd.DataFrame({"median RBC, USD m": np.median(rbc, axis=0) / 1e6,
                          "99.5th pct RBC, USD m": np.quantile(rbc, 0.995, axis=0) / 1e6,
                          "median RBC / live balance %": 100 * np.median(rbc / np.maximum(sim["balance"].sum(axis=2), 1), axis=0)},
                         index=[f"year {t}" for t in range(HORIZON + 1)])
rbc_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
(100 * mix.T).plot(kind="bar", stacked=True, ax=axes[0], colormap="RdYlGn_r", width=0.8)
axes[0].set(title="CM mix of the live book (average path)", xlabel="year", ylabel="% of live balance")
axes[0].legend(fontsize=8, loc="lower right")
bad = np.argsort(rbc[:, -1])[-int(0.005 * N_PATHS):]                            # worst 0.5% of paths by final RBC
bad_mix = [(sim["balance"][bad, t] * (cm_path[bad, t] >= 3)).sum() / sim["balance"][bad, t].sum() for t in range(HORIZON + 1)]
avg_mix = [(sim["balance"][:, t] * (cm_path[:, t] >= 3)).sum() / sim["balance"][:, t].sum() for t in range(HORIZON + 1)]
axes[1].plot(range(HORIZON + 1), 100 * np.array(avg_mix), marker="o", label="average path")
axes[1].plot(range(HORIZON + 1), 100 * np.array(bad_mix), marker="o", color="red", label="worst 0.5% of paths")
axes[1].set(title="Share of live balance in CM3 or worse", xlabel="year", ylabel="%")
axes[1].legend(fontsize=8)
fig.savefig(ROOT / "reports" / "figures" / "cre_cm_migration.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the result:** on the average path the book's category mix barely moves — about 80% CM1 throughout — and required capital falls from USD 1.9m as loans amortise, prepay and mature. In the worst 0.5% of paths the share of the book in CM3 or worse climbs from 5% to 56% by year 5, so the capital charge per dollar rises just as losses arrive. Capital pressure comes from migration in the tail, not from the average.

## 6. Calibration to CRE delinquency history

The property settings (A-07) are declared, not fitted. Two public series test whether they are in a sensible range:

1. **CRE prices** (BIS, quarterly, from 2005): the simulated year-on-year change of the book's property values should swing about as much as the price index.
2. **Bank CRE charge-offs and delinquencies** (from 1991): the simulated annual loss rate should sit below bank levels on average (insurer loans are lower-LTV and better-covered than the average bank CRE loan) but reach a comparable order of magnitude in bad years.

### 6.1 Property values: simulated vs BIS CRE price index

In [ ]:
def load_fred(group, series):
    frame = pd.read_csv(DATA / "raw" / "fred" / group / f"{series}.csv")
    values = pd.to_numeric(frame[series], errors="coerce")
    return pd.Series(values.to_numpy(), index=pd.to_datetime(frame["observation_date"]), name=series).dropna()

bis = load_fred("commercial_real_estate", "COMREPUSQ159N")                     # % change on a year earlier
sim_change = 100 * sim["value_change"][:, 0]                                    # book value index, year 1
pct = [5, 25, 50, 75, 95]
pd.DataFrame({"simulated %": np.r_[np.percentile(sim_change, pct), sim_change.std()],
              "BIS 2005-2026 %": np.r_[np.percentile(bis, pct), bis.std()]},
             index=[f"{p}th pct" for p in pct] + ["std"])

**Reading the result:** the simulated one-year change in the book's property values swings almost exactly as much as the BIS US CRE price index (standard deviation 9.2% vs 9.6%), with a similar bad tail (5th percentile −12.5% vs −11.6%). The BIS sample (2005–2026) is mostly an up-market, so its median growth is higher than the 2% drift assumed here.

### 6.2 Losses: simulated vs bank CRE charge-offs and delinquencies

In [ ]:
charge_off = load_fred("commercial_real_estate", "CORCREXFACBS")
delinquency = load_fred("commercial_real_estate", "DRCRELEXFACBS")
co_year = charge_off.groupby(charge_off.index.year).mean()
dq_year = delinquency.groupby(delinquency.index.year).mean()

loss_rate = 100 * sim["loss"].sum(axis=2) / np.maximum(sim["balance"][:, :-1].sum(axis=2), 1)   # loss / live balance, per year
default_rate = 100 * np.stack([(sim["default_year"] == t).sum(axis=1) / sim["alive"][:, t - 1].sum(axis=1)
                               for t in range(1, HORIZON + 1)], axis=1)
q_worst = 1 - 1 / len(co_year)                                       # 36 years of history: worst year ~ 1 in 36
pd.DataFrame({"simulated annual loss rate %": [loss_rate.mean(), np.median(loss_rate), np.quantile(loss_rate, 0.95), np.quantile(loss_rate, q_worst)],
              "bank charge-off rate %": [co_year.mean(), co_year.median(), co_year.quantile(0.95), co_year.max()],
              "simulated annual default rate %": [default_rate.mean(), np.median(default_rate), np.quantile(default_rate, 0.95), np.quantile(default_rate, q_worst)],
              "bank delinquency rate %": [dq_year.mean(), dq_year.median(), dq_year.quantile(0.95), dq_year.max()]},
             index=["mean", "median", "1 year in 20", "1 year in 36 (bank: worst year since 1991)"])

**Reading the result:** on average the simulated book loses less each year than banks charge off on CRE loans (0.26% vs 0.45%), as expected for lower-LTV insurer loans. In a 1-in-36-year year it loses 2.1%, close to the worst bank year since 1991 (2.35%, in 2009–10). Default rates are not directly comparable — bank *delinquency* counts loans 30+ days late, which is broader than default — but the simulated bad-year default rate sits inside the range of bank delinquency.

## 7. Save outputs

In [ ]:
(DATA / "interim").mkdir(exist_ok=True); (DATA / "processed").mkdir(exist_ok=True)
book.to_parquet(DATA / "interim" / "cre_loans_stylised.parquet")
params = {"settings_A07": SETTINGS, "book_A09": {k: v for k, v in BOOK.items()}, "horizon_years": HORIZON,
          "paths": N_PATHS, "cm_factors": CM_FACTOR, "cycle_link_beta_A14": 0.5}
(DATA / "processed" / "cre_parameters.yaml").write_text(yaml.safe_dump(params, sort_keys=False), encoding="utf-8")
print((DATA / "processed" / "cre_parameters.yaml").read_text()[:400])

## Validation checks

- [x] **CM mapping matches the NAIC grid (V-12)** — every (DSCR, LTV) point falls in exactly one row of Figure 4 and of Figure 5, and spot checks give the category printed in Figure 4.
- [x] **Simulated delinquency vs FRED CRE history** — simulated property-value swings match the BIS price index; the simulated average loss rate sits below the bank average, and the simulated 1-in-36-year loss rate is close to the worst bank year since 1991 (section 6).
- [x] **Balloon default vs Merton closed form** — simulated and closed-form PD agree within simulation error.

In [ ]:
checks = {
    "V-12: NAIC grid has no gaps or overlaps (Figures 4 and 5)": all(c["gaps"] == 0 and c["overlaps"] == 0 for c in coverage.values()),
    "V-12: spot checks match Figure 4": spot_ok,
    "Balloon PD: simulation within 3 standard errors of closed form": abs(pd_sim - pd_closed) < 3 * np.sqrt(pd_sim * (1 - pd_sim) / n_check),
    "CRE value volatility within 25% of BIS index volatility": abs(sim_change.std() / bis.std() - 1) < 0.25,
    "Average simulated loss rate below bank charge-off average": loss_rate.mean() < co_year.mean(),
    "1-in-36-year simulated loss rate within 50% of the worst bank year": abs(np.quantile(loss_rate, q_worst) / co_year.max() - 1) < 0.5,
}
pd.Series(checks).map({True: "PASS", False: "FAIL"})

## Functions to move into the engine

| Function (this notebook) | Target module | Test |
|---|---|---|
| `annual_payment`, `balance_after` | `creditengine.assets.cre` | balance left after 0, 1, 5, 10 years |
| `make_book` (stylised, A-09) | `creditengine.assets.cre` | type mix and averages as configured |
| `cycle_paths`, `step` | `creditengine.simulation.sampling`, `assets.cre` | stationary mean 0, std 1, lag-1 correlation $\phi$ |
| `simulate` (term and balloon default, loss, prepayment) | `creditengine.assets.cre` | balloon PD vs Merton closed form |
| `GRID`, `cm_category` (table-driven) | `creditengine.assets.cre` — **refactor** | no gaps or overlaps; Figure 4 spot checks |
| CRE calibration checks | `creditengine.validation` — **new** | value swings vs BIS; loss rates vs bank charge-offs |

**refactor** = the engine codes the grid as a sequence of if-rules; a typed table is easier to check against the NAIC document line by line.

## Findings and next steps

**Key results**
- **Defaults:** 3.1% of loans default within five years; 95% of them at the balloon. The double trigger (DSCR < 1 **and** LTV > 1) makes term default rare for a book with DSCR around 2.
- **Losses:** mean LGD 29% with κ = 25%; five-year EL 0.88% of the book, VaR 99.5% 7.5%, ES 99.5% 9.7%.
- **Capital:** year-0 RBC USD 1.9m (1.1% of balance); on average it falls as the book amortises, but in the worst 0.5% of paths the CM3-or-worse share rises from 5% to 56%.
- **Calibration:** property-value swings match the BIS index (std 9.2% vs 9.6%); the 1-in-36-year loss rate 2.1% vs worst bank year 2.35%; average loss rate below banks (0.26% vs 0.45%).
- **NAIC grid:** typed line by line from Figures 4 and 5; covers every (DSCR, LTV) point exactly once. One printed line in Figure 5 (CM5) only makes sense as DSC < 1.10.

**Assumptions** (record in `docs/assumption_register.md`)
- A-07 unchanged: value drift 2%, vol 15%, cycle loading 0.35; NOI growth 2%, vol 8%, loading 0.30; κ = 25%; prepayment 3%; refinance at 6% needs LTV ≤ 100% and DSCR ≥ 1.
- A-09: stylised 60-loan book; A-14: CRE cycle half the global cycle in the whole-book run.
- Added: Figure 5 CM5 line read as DSC < 1.10.

**Limitations** (record in `docs/limitations.md`)
1. **Stylised book, not loan-level data:** CMBS (Form ABS-EE) and Fannie Mae MFLPD loan data are planned for v0.3 to fit the property settings rather than declare them.
2. **No extension or modification:** a loan that fails refinancing defaults immediately; in practice many are extended, which delays and often reduces loss.
3. **CM6/CM7 not tracked:** a defaulted loan leaves the book at once instead of passing through 90-days-overdue and foreclosure categories with 18% and 23% factors.
4. **Fixed refinance rate (6%):** the rate at maturity should move with interest rates; a rate shock is the main driver of the recent maturity wall.
5. **Small book:** 60 loans make annual loss rates lumpy compared with aggregate bank data.

**Next:** NB06 (corporate bond portfolio Monte Carlo), then NB03 (credit cycle).